In [ ]:
import os
import cv2
import shutil

# --- 1. YOUR EXACT PATHS ---
images_dir = '/kaggle/input/notebooks/adenijiyusuf/detection-model/Corrosion-Detection-1/train/images'
labels_dir = '/kaggle/input/notebooks/adenijiyusuf/detection-model/Corrosion-Detection-1/train/labels'

output_dir = '/kaggle/working/all_rust_crops'
os.makedirs(output_dir, exist_ok=True)

print("Verifying your folders...")
if not os.path.exists(images_dir) or not os.path.exists(labels_dir):
    print(" Wait! Kaggle can't find those exact folders. Double-check that the spelling matches your Input menu.")
else:
    print(f" Found Images: {images_dir}")
    print(f" Found Labels: {labels_dir}")
    print("\n Starting the cropping process...")

    crop_count = 0

    # --- 2. THE CUTTING LOOP ---
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith('.txt'):
            continue

        image_file = label_file.replace('.txt', '.jpg') 
        image_path = os.path.join(images_dir, image_file)
        label_path = os.path.join(labels_dir, label_file)

        if not os.path.exists(image_path):
            continue

        img = cv2.imread(image_path)
        if img is None:
            continue
        img_height, img_width, _ = img.shape

        with open(label_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                x_center, y_center, box_width, box_height = map(float, parts[1:5])

                xmin = int((x_center - box_width / 2) * img_width)
                xmax = int((x_center + box_width / 2) * img_width)
                ymin = int((y_center - box_height / 2) * img_height)
                ymax = int((y_center + box_height / 2) * img_height)

                xmin, ymin = max(0, xmin), max(0, ymin)
                xmax, ymax = min(img_width, xmax), min(img_height, ymax)

                cropped_img = img[ymin:ymax, xmin:xmax]

                if cropped_img.size > 0:
                    crop_count += 1
                    save_path = os.path.join(output_dir, f'rust_crop_{crop_count}.jpg')
                    cv2.imwrite(save_path, cropped_img)

    print(f"\n Success! Cut out {crop_count} individual rust patches.")

    # --- 3. BUNDLE FOR DOWNLOAD ---
    print(" Zipping the folder for you...")
    shutil.make_archive('/kaggle/working/all_rust_crops', 'zip', output_dir)
    print(" Done! Look in your right-hand Kaggle Output menu for 'all_rust_crops.zip' and download it to your laptop.")

In [1]:
import os
import cv2
import random
import shutil

# --- PATHS ---
images_dir = '/kaggle/input/notebooks/adenijiyusuf/detection-model/Corrosion-Detection-1/train/images'
labels_dir = '/kaggle/input/notebooks/adenijiyusuf/detection-model/Corrosion-Detection-1/train/labels'
output_dir = '/kaggle/working/auto_grade_0'

os.makedirs(output_dir, exist_ok=True)

TARGET_CROPS = 500  # Adjust this number based on how many Grade 0 images you need
CROP_SIZE = 150     # Pixel size of the square cut

clean_count = 0
image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]

print(f"Hunting for {TARGET_CROPS} clean steel patches...")

while clean_count < TARGET_CROPS:
    image_file = random.choice(image_files)
    image_path = os.path.join(images_dir, image_file)
    label_path = os.path.join(labels_dir, image_file.replace('.jpg', '.txt').replace('.png', '.txt'))
    
    img = cv2.imread(image_path)
    if img is None:
        continue
        
    h, w, _ = img.shape
    if h <= CROP_SIZE or w <= CROP_SIZE:
        continue 

    rust_boxes = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    x_center, y_center, box_width, box_height = map(float, parts[1:5])
                    xmin = int((x_center - box_width / 2) * w)
                    xmax = int((x_center + box_width / 2) * w)
                    ymin = int((y_center - box_height / 2) * h)
                    ymax = int((y_center + box_height / 2) * h)
                    rust_boxes.append((xmin, ymin, xmax, ymax))

    for _ in range(10):
        if clean_count >= TARGET_CROPS:
            break
            
        c_xmin = random.randint(0, w - CROP_SIZE)
        c_ymin = random.randint(0, h - CROP_SIZE)
        c_xmax = c_xmin + CROP_SIZE
        c_ymax = c_ymin + CROP_SIZE
        
        hits_rust = False
        for r_xmin, r_ymin, r_xmax, r_ymax in rust_boxes:
            if not (c_xmax < r_xmin or c_xmin > r_xmax or c_ymax < r_ymin or c_ymin > r_ymax):
                hits_rust = True
                break
                
        if not hits_rust:
            clean_crop = img[c_ymin:c_ymax, c_xmin:c_xmax]
            save_path = os.path.join(output_dir, f'clean_steel_{clean_count}.jpg')
            cv2.imwrite(save_path, clean_crop)
            clean_count += 1

print("Zipping the folder...")
shutil.make_archive('/kaggle/working/auto_grade_0', 'zip', output_dir)
print("Done! Check your Kaggle output for 'auto_grade_0.zip'.")

Hunting for 500 clean steel patches...
Zipping the folder...
Done! Check your Kaggle output for 'auto_grade_0.zip'.


In [1]:
import os
import cv2
import random

# --- 1. SET YOUR FOLDER PATH ---
source_folder = '/kaggle/input/datasets/adenijiyusuf/grade-0/grade_0' 

# Grab ALL images, whether they are jpg, jpeg, or png
images = [
    os.path.join(source_folder, f) 
    for f in os.listdir(source_folder) 
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

current_count = len(images)
target_count = 415

print(f"Found {current_count} mixed-format images. Multiplying up to {target_count}...")

if current_count == 0:
    print(" Wait! No images found. Check your source_folder path.")
else:
    # --- 2. THE CLONING LOOP ---
    while current_count < target_count:
        # Pick a random image from your original batch
        img_path = random.choice(images)
        img = cv2.imread(img_path)

        if img is None:
            continue

        # Randomly choose an augmentation trick
        trick = random.randint(1, 3)

        if trick == 1:
            # Mirror it horizontally
            aug_img = cv2.flip(img, 1)
            suffix = "_hflip"
        elif trick == 2:
            # Flip it upside down
            aug_img = cv2.flip(img, 0)
            suffix = "_vflip"
        else:
            # Rotate it 90 degrees
            aug_img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
            suffix = "_rot90"

        # Save the new image uniformly as a .jpg
        base_name = os.path.basename(img_path).split('.')[0]
        new_name = f"{base_name}{suffix}_{current_count}.jpg"
        save_path = os.path.join(source_folder, new_name)

        cv2.imwrite(save_path, aug_img)
        current_count += 1

    print(f"You now have exactly {current_count} Grade 0 images ready for training.")

Found 149 mixed-format images. Multiplying up to 415...
You now have exactly 415 Grade 0 images ready for training.


In [2]:
import os
print(os.listdir('/kaggle/working'))

['auto_grade_0.zip', 'all_rust_crops.zip', '.virtual_documents', 'auto_grade_0', 'state.db', 'all_rust_crops']


In [3]:
import os
from IPython.display import FileLink

# 1. First, let's print the files just to prove to you it's actually there
print("Files hiding in Kaggle's brain:")
print(os.listdir('/kaggle/working'))

# 2. Generate a magic download button
print("\n Click the blue link below to download your data ")
FileLink(r'grade_0_final.zip')

Files hiding in Kaggle's brain:
['auto_grade_0.zip', 'all_rust_crops.zip', '.virtual_documents', 'auto_grade_0', 'state.db', 'all_rust_crops']

 Click the blue link below to download your data 


/kaggle/working/grade_0_final.zip

In [7]:
import os
import cv2
import random
import shutil
from IPython.display import display, HTML

# --- 1. PASTE YOUR EXACT INPUT PATH HERE ---
# (Use the exact path that successfully found your 149 images earlier)
input_path = '/kaggle/input/datasets/adenijiyusuf/grade-0/grade_0' 

# We will build everything safely in this brand new writable folder
safe_folder = '/kaggle/working/guaranteed_grade_0'
zip_name = 'guaranteed_grade_0'
zip_path = f'/kaggle/working/{zip_name}.zip'

print(" Initiating God Mode Pipeline...\n")

# Clear old attempts so we start fresh
if os.path.exists(safe_folder):
    shutil.rmtree(safe_folder)
os.makedirs(safe_folder, exist_ok=True)

# --- 2. SAFELY COPY ORIGINALS OUT OF READ-ONLY JAIL ---
try:
    original_images = [f for f in os.listdir(input_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(original_images) == 0:
        print(f" ERROR: I looked in '{input_path}' but couldn't find any images. Did you paste the exact path?")
        raise Exception("No images found.")
    
    print(f" Found {len(original_images)} images. Safely copying them to Kaggle's writable memory...")
    for img in original_images:
        shutil.copy(os.path.join(input_path, img), os.path.join(safe_folder, img))
        
except FileNotFoundError:
    print(f" ERROR: The path '{input_path}' does not exist. Double check your copy-paste!")
    raise Exception("Path not found.")

# --- 3. MULTIPLY IN THE SAFE FOLDER ---
working_images = [os.path.join(safe_folder, f) for f in os.listdir(safe_folder)]
current_count = len(working_images)
target_count = 415

print(f"\n Multiplying {current_count} images up to {target_count}...")

while current_count < target_count:
    img_path = random.choice(working_images)
    img = cv2.imread(img_path)
    if img is None: continue

    trick = random.randint(1, 3)
    if trick == 1: aug_img = cv2.flip(img, 1)
    elif trick == 2: aug_img = cv2.flip(img, 0)
    else: aug_img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)

    new_name = f"aug_{current_count}.jpg"
    save_path = os.path.join(safe_folder, new_name)
    
    # This write will actually work now!
    cv2.imwrite(save_path, aug_img)
    current_count += 1

print(f" Multiplication complete. You have exactly {len(os.listdir(safe_folder))} actual files saved.")

# --- 4. ZIP AND GENERATE BUTTON ---
print("\n Zipping the folder right now...")
shutil.make_archive(f'/kaggle/working/{zip_name}', 'zip', safe_folder)

if os.path.exists(zip_path):
    print(" Zip created successfully! Here is your magic link:")
    download_link = f'<a href="./{zip_name}.zip" target="_blank" style="font-size: 24px; font-weight: bold; color: #1a73e8; text-decoration: underline;"> CLICK HERE TO DOWNLOAD YOUR 415 IMAGES </a>'
    display(HTML(download_link))
else:
    print(" Kaggle UI completely blocked the zip creation.")

 Initiating God Mode Pipeline...

 Found 149 images. Safely copying them to Kaggle's writable memory...

 Multiplying 149 images up to 415...
 Multiplication complete. You have exactly 415 actual files saved.

 Zipping the folder right now...
 Zip created successfully! Here is your magic link:


In [2]:
# Install the required libraries first!
!pip install split-folders ultralytics

import splitfolders
from ultralytics import YOLO

# --- 1. SPLIT THE DATA ---
input_data_path = "/kaggle/input/datasets/adenijiyusuf/corrosion-grader/graded_corrosion" # Make sure this matches your dataset name!
output_dataset_path = "/kaggle/working/classification_dataset"

print(" Splitting data into 80% Train / 20% Val...")
splitfolders.ratio(
    input_data_path, 
    output=output_dataset_path, 
    seed=42, 
    ratio=(0.8, 0.2), 
    group_prefix=None
)

# --- 2. TRAIN THE GRADER ---
print(" Loading YOLO Classification model...")
grader_model = YOLO('yolov8n-cls.pt') 

print(" Starting 50-Epoch Training...")
results = grader_model.train(
    data=output_dataset_path, 
    epochs=50,                
    imgsz=224,                
    batch=32,
    device=0                  
)
print("Done! Download your new 'best.pt' from the runs folder!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.4 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 Splitting data into 80% Train / 20% Val...


Copying files: 1643 files [00:06, 240.54 files/s]


 Loading YOLO Classification model...
 Starting 50-Epoch Training...
Ultralytics 8.4.39  Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/classification_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False